# ROAM — fine-tune DocLayout-YOLO on land-record layout classes

Fine-tunes the pretrained `DocLayout-YOLO-DocStructBench` model on our 6 hand-labeled
classes: `Text`, `Table`, `Picture`, `Seal`, `ParcelMap`, `ScannedPrintout`.

**Before running:** attach the `roam-finetune-dataset` dataset (the zip you uploaded)
to this notebook via the Kaggle sidebar (Add Input), and turn on a GPU
(Settings → Accelerator → GPU T4 x2, or any available GPU).

In [ ]:
!pip install -q doclayout-yolo huggingface_hub

In [ ]:
import os

# doclayout-yolo auto-registers callbacks for several optional third-party
# integrations (Weights & Biases, Ray Tune, ClearML, Comet, ...) whenever
# the corresponding package happens to be importable in the environment --
# Kaggle ships several of these by default. Two of them are broken in this
# combination of package versions:
#   - wandb: tries to use our filesystem output path as its project name,
#     which wandb rejects (project names can't contain '/').
#   - ray: its on_fit_epoch_end callback calls ray.tune.is_session_enabled(),
#     which doesn't exist in the Ray version Kaggle has installed (renamed/
#     removed upstream) -- crashes at the end of every epoch.
# We don't use either integration -- everything is saved locally and zipped
# at the end -- so the reliable fix is to remove both packages, which stops
# doclayout-yolo from registering their callbacks in the first place.
os.environ["WANDB_MODE"] = "disabled"
!pip uninstall -y wandb ray -q

In [ ]:
import torch

# PyTorch 2.6 changed torch.load's default from weights_only=False to
# weights_only=True for security. doclayout-yolo's internal checkpoint
# utilities (e.g. strip_optimizer, called automatically at the end of
# training) predate that change and don't pass weights_only explicitly,
# so they crash trying to unpickle the custom model class in our own
# checkpoint. This restores the old default globally for this session.
# Safe here specifically because every file loaded this way is one this
# same training run just created -- never an untrusted external download.
_original_torch_load = torch.load


def _torch_load_weights_only_false(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _original_torch_load(*args, **kwargs)


torch.load = _torch_load_weights_only_false

In [ ]:
import shutil
from pathlib import Path

# Kaggle mounts uploaded datasets read-only under /kaggle/input/<dataset-name>/.
# Adjust DATASET_NAME if you named the uploaded dataset something else.
DATASET_NAME = "roam-finetune-dataset"
SRC = Path(f"/kaggle/input/{DATASET_NAME}")
WORK = Path("/kaggle/working/finetune_dataset")

if not SRC.exists():
    raise FileNotFoundError(
        f"{SRC} not found — attach the dataset via Add Input first, "
        "and confirm its exact name matches DATASET_NAME above."
    )

# Kaggle input is read-only; copy to /kaggle/working so training can write
# cache files (*.cache) next to the label folders.
if WORK.exists():
    shutil.rmtree(WORK)
shutil.copytree(SRC, WORK)

print("Dataset staged at", WORK)
print("train images:", len(list((WORK / "images/train").glob("*.png"))))
print("val images:", len(list((WORK / "images/val").glob("*.png"))))

In [ ]:
# data.yaml ships with `path: .` (relative, for local use). Point it at the
# staged copy here so Kaggle resolves images/train, images/val correctly.
data_yaml = WORK / "data.yaml"
text = data_yaml.read_text()
text = text.replace("path: .", f"path: {WORK}")
data_yaml.write_text(text)
print(data_yaml.read_text())

In [ ]:
from huggingface_hub import hf_hub_download

weights_path = hf_hub_download(
    repo_id="juliozhao/DocLayout-YOLO-DocStructBench",
    filename="doclayout_yolo_docstructbench_imgsz1024.pt",
)
print("Pretrained weights:", weights_path)

## Fine-tune

Starts from the pretrained DocStructBench checkpoint. Since our 6 classes differ
from DocLayNet's 11, the detection head is reinitialized automatically; the
backbone stays initialized from the pretrained weights (transfer learning).

`freeze=10` locks the first 10 layers (the backbone) so the small fine-tune
set doesn't overwrite the general layout features already learned — reduces
overfitting risk given our dataset size.

In [ ]:
from doclayout_yolo import YOLOv10

model = YOLOv10(weights_path)

results = model.train(
    data=str(data_yaml),
    epochs=100,
    imgsz=1024,
    # Training in FP32 (amp=False, see below) uses roughly 2x the activation
    # memory of mixed precision. batch=16 at imgsz=1024 in FP32 was right at
    # the T4's 14.56GB ceiling and OOM'd. batch=8 gives comfortable headroom;
    # drop to batch=4 if this still OOMs on your GPU.
    batch=8,
    device=0,
    freeze=10,
    patience=20,
    project="/kaggle/working/runs",
    name="roam_finetune",
    exist_ok=True,
    # AMP's pre-flight sanity check tries to download a reference yolov8n.pt
    # from a broken URL baked into this package version (points at a GitHub
    # org/repo that doesn't exist) and crashes when the download silently
    # fails. Disabling AMP skips that check entirely -- trains in FP32
    # instead of mixed precision, at the cost of the extra memory this cell
    # is now compensating for via the smaller batch size.
    amp=False,
)

## Evaluate

Same rigor as the rest of this project: check recall specifically, per class —
`ParcelMap` recall matters most (a missed map page is the costliest error).

In [ ]:
run_dir = Path("/kaggle/working/runs/roam_finetune")
best_weights = run_dir / "weights" / "best.pt"

metrics = model.val(data=str(data_yaml), imgsz=1024, device=0)

class_names = ["Text", "Table", "Picture", "Seal", "ParcelMap", "ScannedPrintout"]
print(f"{'class':16s} {'precision':>10s} {'recall':>10s} {'mAP50':>10s}")
for i, name in enumerate(class_names):
    p = metrics.box.p[i] if i < len(metrics.box.p) else float("nan")
    r = metrics.box.r[i] if i < len(metrics.box.r) else float("nan")
    ap50 = metrics.box.ap50[i] if i < len(metrics.box.ap50) else float("nan")
    print(f"{name:16s} {p:>10.3f} {r:>10.3f} {ap50:>10.3f}")

print()
print("mAP50 (all classes):", metrics.box.map50)
print("mAP50-95 (all classes):", metrics.box.map)

## Package outputs for download

Zips the fine-tuned weights, training curves, confusion matrix, and a copy of
this run's metrics into one file under `/kaggle/working/` — download it from
the notebook's Output tab after the run finishes.

In [ ]:
import json
import shutil

output_dir = Path("/kaggle/working/roam_finetune_output")
if output_dir.exists():
    shutil.rmtree(output_dir)
output_dir.mkdir()

# Weights
shutil.copy(best_weights, output_dir / "roam_layout_best.pt")
shutil.copy(run_dir / "weights" / "last.pt", output_dir / "roam_layout_last.pt")

# Training plots ultralytics/doclayout-yolo writes automatically
for fname in [
    "results.png", "results.csv", "confusion_matrix.png",
    "confusion_matrix_normalized.png", "PR_curve.png", "F1_curve.png",
    "labels.jpg", "args.yaml",
]:
    src = run_dir / fname
    if src.exists():
        shutil.copy(src, output_dir / fname)

# Per-class metrics summary as JSON, for quick reference without re-running
summary = {
    "class_names": class_names,
    "per_class": {
        name: {
            "precision": float(metrics.box.p[i]) if i < len(metrics.box.p) else None,
            "recall": float(metrics.box.r[i]) if i < len(metrics.box.r) else None,
            "mAP50": float(metrics.box.ap50[i]) if i < len(metrics.box.ap50) else None,
        }
        for i, name in enumerate(class_names)
    },
    "map50": float(metrics.box.map50),
    "map50_95": float(metrics.box.map),
}
(output_dir / "metrics_summary.json").write_text(json.dumps(summary, indent=2))

zip_path = shutil.make_archive("/kaggle/working/roam_finetune_output", "zip", output_dir)
print("Ready to download:", zip_path)